In [74]:
import pdfplumber
import re

pattern = r'([A-Za-z0-9 &\-"]+)\n\s*\n\s*([\d,]+)\nABOUT THIS PRODUCT.*?(?=\n[A-Za-z0-9 &\-"]+\n\s*\n\s*[\d,]+\nABOUT THIS PRODUCT|\Z)'
pages_data = []


with pdfplumber.open("product_res.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        text = re.sub(r'\(cid:\d+\)', '', text)
        text = re.sub(r'\n+', '\n', text)
        text = text.replace("\nn\n", "\n") 
        products = re.findall(pattern, text, re.DOTALL)
        pages_data.append({ "text": text, })

In [75]:
pages_data

[{'text': 'n\nELECTRONICS\nProVision X15 Laptop\n89,999\nABOUT THIS PRODUCT\nA powerhouse ultrabook engineered for professionals who demand performance without\ncompromise. Razor-thin chassis meets all-day battery endurance.\nPRODUCT DESCRIPTION\nThe ProVision X15 redefines mobile productivity. Its 12th-gen Intel Core i7 processor paired with LPDDR5\nRAM ensures snappy multitasking, while the OLED display delivers cinema-grade colour accuracy. The\nMIL-STD-810H rated chassis weighs just 1.3 kg, making it the ideal travel companion for designers,\ndevelopers, and executives alike.\nPRODUCT SPECIFICATIONS\nSpecification Details\nProcessor Intel Core i7-1260P (12-core)\nRAM 32 GB LPDDR5\nStorage 1 TB NVMe PCIe Gen 4 SSD\nDisplay 14" 2.8K OLED, 120 Hz\nBattery 72 Wh, up to 14 hrs\nWeight 1.3 kg\nOS Windows 11 Pro\nPorts 2x USB-C, 2x USB-A, HDMI 2.1, SD Card\nSoundWave Pro ANC Headphones\n14,499\nABOUT THIS PRODUCT\nIndustry-leading active noise cancellation wrapped in premium leather cushi

In [76]:
import pdfplumber
import re
import uuid

pattern = r'([A-Za-z0-9 &\-"]+)\n\s*\n\s*([\d,]+)\nABOUT THIS PRODUCT.*?(?=\n[A-Za-z0-9 &\-"]+\n\s*\n\s*[\d,]+\nABOUT THIS PRODUCT|\Z)'

pages_data = []
all_products = []


In [77]:
def parse_specs(specs_text):
    specs = {}

    lines = specs_text.split("\n")

    for line in lines:
        line = line.strip()

        # skip headers
        if not line or "Specification" in line:
            continue

        # split key-value
        parts = line.split(" ", 1)

        if len(parts) == 2:
            key, value = parts
            specs[key.strip()] = value.strip()

    return specs

In [87]:
def extract_sections(content):

    about = ""
    description = ""
    specs_text = ""

    about_match = re.search(
        r'ABOUT THIS PRODUCT\n(.*?)\nPRODUCT DESCRIPTION',
        content,
        re.DOTALL
    )
    if about_match:
        about = about_match.group(1).strip()

    desc_match = re.search(
        r'PRODUCT DESCRIPTION\n(.*?)\nPRODUCT SPECIFICATIONS',
        content,
        re.DOTALL
    )
    if desc_match:
        description = desc_match.group(1).strip()

    specs_match = re.search(
        r'PRODUCT SPECIFICATIONS\n(.*)',
        content,
        re.DOTALL
    )
    if specs_match:
        specs_text = specs_match.group(1).strip()

    return about, description, specs_text
# parse_specs(specs_text)

In [92]:
current_product = None

with pdfplumber.open("product_res.pdf") as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()

        if not text:
            continue
        text = re.sub(r'\(cid:\d+\)', '', text)
        text = re.sub(r'\n+', '\n', text)

        lines = text.split("\n")


        i = 0
        while i < len(lines):
            line = lines[i].strip()

            if "thank you for exploring" in line.lower():
                break

            if (
                i + 2 < len(lines)
                and lines[i+1].strip() == "n"
                and re.match(r'[\d,]+', lines[i+2].strip())
            ):
                if current_product:
                    all_products.append(current_product)

                current_product = {
                    "id": str(uuid.uuid4()),
                    "page": page_num,
                    "name": line,
                    "price": lines[i+2].strip(),
                    "content": ""
                }
                i += 3
                continue
            
            if current_product:
                current_product["content"] += line + "\n"

            i += 1

        if current_product:
            all_products.append(current_product)

        

In [93]:
all_products

[{'id': 'ad0eb0c4-b225-45b2-b3fd-7558d121d145',
  'page': 1,
  'name': 'ProVision X15 Laptop',
  'price': '89,999',
  'content': 'ABOUT THIS PRODUCT\nA powerhouse ultrabook engineered for professionals who demand performance without\ncompromise. Razor-thin chassis meets all-day battery endurance.\nPRODUCT DESCRIPTION\nThe ProVision X15 redefines mobile productivity. Its 12th-gen Intel Core i7 processor paired with LPDDR5\nRAM ensures snappy multitasking, while the OLED display delivers cinema-grade colour accuracy. The\nMIL-STD-810H rated chassis weighs just 1.3 kg, making it the ideal travel companion for designers,\ndevelopers, and executives alike.\nPRODUCT SPECIFICATIONS\nSpecification Details\nProcessor Intel Core i7-1260P (12-core)\nRAM 32 GB LPDDR5\nStorage 1 TB NVMe PCIe Gen 4 SSD\nDisplay 14" 2.8K OLED, 120 Hz\nBattery 72 Wh, up to 14 hrs\nWeight 1.3 kg\nOS Windows 11 Pro\nPorts 2x USB-C, 2x USB-A, HDMI 2.1, SD Card\n'},
 {'id': 'f7a07b01-3312-49fd-a66a-0edd361d2583',
  'page'

In [95]:
result = []
for product in all_products:
    about, description, specs_text = extract_sections(product['content'])
    res = parse_specs(specs_text)
    result.append({
        **product, "about": about, "description": description, "specification": res
    })

In [96]:
result

[{'id': 'ad0eb0c4-b225-45b2-b3fd-7558d121d145',
  'page': 1,
  'name': 'ProVision X15 Laptop',
  'price': '89,999',
  'content': 'ABOUT THIS PRODUCT\nA powerhouse ultrabook engineered for professionals who demand performance without\ncompromise. Razor-thin chassis meets all-day battery endurance.\nPRODUCT DESCRIPTION\nThe ProVision X15 redefines mobile productivity. Its 12th-gen Intel Core i7 processor paired with LPDDR5\nRAM ensures snappy multitasking, while the OLED display delivers cinema-grade colour accuracy. The\nMIL-STD-810H rated chassis weighs just 1.3 kg, making it the ideal travel companion for designers,\ndevelopers, and executives alike.\nPRODUCT SPECIFICATIONS\nSpecification Details\nProcessor Intel Core i7-1260P (12-core)\nRAM 32 GB LPDDR5\nStorage 1 TB NVMe PCIe Gen 4 SSD\nDisplay 14" 2.8K OLED, 120 Hz\nBattery 72 Wh, up to 14 hrs\nWeight 1.3 kg\nOS Windows 11 Pro\nPorts 2x USB-C, 2x USB-A, HDMI 2.1, SD Card\n',
  'about': 'A powerhouse ultrabook engineered for profess

In [100]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams

client = QdrantClient(
    url="http://localhost:6333"
)

In [101]:
client

In [102]:
client.recreate_collection(
    collection_name="products",
    vectors_config={
        "dense": VectorParams(size=384, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

/tmp/ipykernel_3021499/2336667356.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [103]:
from fastembed import TextEmbedding, SparseTextEmbedding

dense_model = TextEmbedding("BAAI/bge-small-en")
sparse_model = SparseTextEmbedding("Qdrant/bm25")

Fetching 18 files: 100%|██████████| 18/18 [00:03<00:00,  5.68it/s]


In [105]:
product = result[0]

In [111]:
def build_text(product):
    specs_text = " ".join([f"{k}: {v}" for k, v in product["specification"].items()])
    
    return f"""
    {product['name']}
    {product['about']}
    {product['description']}
    {specs_text}
    """

dense_vector = list(dense_model.embed([text]))[0].tolist()
sparse_vector = list(sparse_model.embed([text]))[0]

In [108]:
text = build_text(product)

In [113]:
from qdrant_client.models import PointStruct

points = []

dense_vector = list(dense_model.embed([text]))[0].tolist()
sparse_vector = list(sparse_model.embed([text]))[0]

points.append(
    PointStruct(
        id=product["id"],
        vector={
            "dense": dense_vector,
            "sparse": sparse_vector
        },
        payload={
            "name": product["name"],
            "price": product["price"],
            "page": product["page"],
            "about": product["about"],
            "description": product["description"],
            "specification": product["specification"]
        }
    )
)

ValidationError: 20 validation errors for PointStruct
vector.list[float]
  Input should be a valid list [type=list_type, input_value={'dense': [-0.01332106906...4,
        245350191]))}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type
vector.list[list[float]]
  Input should be a valid list [type=list_type, input_value={'dense': [-0.01332106906...4,
        245350191]))}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type
vector.dict[str,union[list[float],SparseVector,list[list[float]],Document,Image,InferenceObject]].sparse.list[float]
  Input should be a valid list [type=list_type, input_value=SparseEmbedding(values=ar...74,
        245350191])), input_type=SparseEmbedding]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type
vector.dict[str,union[list[float],SparseVector,list[list[float]],Document,Image,InferenceObject]].sparse.SparseVector
  Input should be a valid 